In [1]:
# ResNet trained from scratch on permitted competition data.
# References:
# ResNet: https://arxiv.org/abs/1512.03385
# Random Erasing: https://arxiv.org/abs/1708.04896
# MixUp: https://arxiv.org/abs/1710.09412

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
import random
import json
import time
import hashlib

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split
import kagglehub

# Settings
SEED = 42
EPOCHS = 200
BATCH = 256
WIDTHS = [64, 128, 256]
BLOCKS = [3, 3, 3]
MIXUP_ALPHA = 0.2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)

DATA = Path(
    kagglehub.competition_download(
        "cifar-10-res-net-architecture-challenge-dl"
    )
)

# Each execution gets a separate folder.
run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT = Path("/kaggle/working") / f"resnet_3stage_mixup_{run_name}"
OUT.mkdir(parents=True, exist_ok=False)

print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("Data:", DATA, flush=True)
print("Output folder:", OUT, flush=True)

# Read competition metadata.
train = pd.read_csv(DATA / "train_labels.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")

assert len(train) == 50000
assert train.label.between(0, 9).all()
assert train.id.is_unique and train.filename.is_unique
assert test.id.is_unique and test.filename.is_unique
assert sample.id.is_unique
assert sample.columns.tolist() == ["id", "label"]
assert len(sample) == len(test)
assert set(sample.id) == set(test.id)
assert not train.isna().any().any()
assert not test.isna().any().any()


def load_images(frame, folder):
    def read_image(filename):
        with Image.open(DATA / folder / filename) as image:
            array = np.array(image.convert("RGB"))
        assert array.shape == (32, 32, 3)
        return array

    with ThreadPoolExecutor(max_workers=8) as pool:
        return np.stack(list(pool.map(read_image, frame.filename)))


print("Loading permitted images into memory...", flush=True)
images = load_images(train, "train_images")
test_images = load_images(test, "test_images")
labels = train.label.to_numpy(dtype=np.int64)

# Preserve the same split for comparison with previous experiments.
tr, va = train_test_split(
    np.arange(len(train)),
    test_size=5000,
    stratify=labels,
    random_state=SEED,
)
np.savez(OUT / "split_seed42.npz", train=tr, validation=va)

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

aug = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean, std),
    T.RandomErasing(
        p=0.5,
        scale=(0.02, 0.20),
        ratio=(0.3, 3.3),
        value=0,
    ),
])

plain = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])


class Images(Dataset):
    def __init__(self, x, y=None, transform=plain):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        x = self.transform(Image.fromarray(self.x[index]))
        if self.y is None:
            return x
        return x, int(self.y[index])


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )


train_dl = make_loader(
    Images(images[tr], labels[tr], aug),
    shuffle=True,
)
val_dl = make_loader(Images(images[va], labels[va]))
test_dl = make_loader(Images(test_images))


class Block(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(cin, cout, 3, stride, 1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False),
            nn.BatchNorm2d(cout),
        )
        if cin == cout and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False),
                nn.BatchNorm2d(cout),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.f(x) + self.skip(x))


class SmallResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, WIDTHS[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(WIDTHS[0]),
            nn.ReLU(inplace=True),
        )

        layers = []
        cin = WIDTHS[0]

        for stage, (cout, count) in enumerate(zip(WIDTHS, BLOCKS)):
            for block_index in range(count):
                stride = 2 if stage > 0 and block_index == 0 else 1
                layers.append(Block(cin, cout, stride))
                cin = cout

        self.body = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(WIDTHS[-1], 10)

    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


# Fresh initialization: no pretrained weights.
model = SmallResNet().to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert params == 4_327_754
assert params <= 5_000_000
print("Trainable parameters:", params, flush=True)

config = {
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH,
    "widths": WIDTHS,
    "blocks": BLOCKS,
    "parameters": params,
    "pretrained": False,
    "optimizer": "SGD",
    "learning_rate": 0.1,
    "momentum": 0.9,
    "nesterov": True,
    "weight_decay": 5e-4,
    "label_smoothing": 0.1,
    "scheduler": "CosineAnnealingLR",
    "scheduler_t_max": EPOCHS,
    "mixup_alpha": MIXUP_ALPHA,
    "validation_size": len(va),
    "normalization_mean": mean,
    "normalization_std": std,
    "random_crop_padding": 4,
    "horizontal_flip_probability": 0.5,
    "random_erasing": {
        "p": 0.5,
        "scale": [0.02, 0.20],
        "ratio": [0.3, 3.3],
        "value": 0,
    },
    "checkpoint_selection": "highest raw validation accuracy",
    "cudnn_benchmark": True,
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
}
(OUT / "config.json").write_text(json.dumps(config, indent=2))

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda")

best = -1.0
best_epoch = 0
history = []

for epoch in range(EPOCHS):
    started = time.time()
    model.train()
    total_loss = 0.0
    weighted_correct = 0.0
    total = 0

    for x, y in train_dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # Mix only examples from the training split.
        if MIXUP_ALPHA > 0:
            lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
        else:
            lam = 1.0

        perm = torch.randperm(len(y), device=device)
        mixed_x = lam * x + (1.0 - lam) * x[perm]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(mixed_x)
            loss = (
                lam * criterion(logits, y)
                + (1.0 - lam) * criterion(logits, y[perm])
            )

        assert torch.isfinite(loss).item(), "Non-finite training loss."
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * len(y)
        predicted = logits.argmax(1)
        weighted_correct += (
            lam * (predicted == y).sum().item()
            + (1.0 - lam) * (predicted == y[perm]).sum().item()
        )
        total += len(y)

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for x, y in val_dl:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
            val_correct += (logits.argmax(1) == y).sum().item()
            val_total += len(y)

    val_acc = val_correct / val_total
    row = {
        "epoch": epoch + 1,
        "loss": total_loss / total,
        "mixup_train_accuracy": weighted_correct / total,
        "val_accuracy": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
        "seconds": time.time() - started,
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUT / "training_log.csv", index=False)

    if val_acc > best:
        best = val_acc
        best_epoch = epoch + 1
        torch.save({
            "model": model.state_dict(),
            "config": config,
            "val_accuracy": best,
            "epoch": best_epoch,
        }, OUT / "best.pt")

    scheduler.step()
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": epoch + 1,
        "best_val_accuracy": best,
        "best_epoch": best_epoch,
        "config": config,
    }, OUT / "last.pt")

    print(
        f"Epoch {epoch + 1}/{EPOCHS}: "
        f"loss={row['loss']:.4f} "
        f"mixup_train={row['mixup_train_accuracy']:.4f} "
        f"val={val_acc:.4f} "
        f"best={best:.4f} "
        f"seconds={row['seconds']:.0f}",
        flush=True,
    )

# Load only this run's best checkpoint.
checkpoint = torch.load(
    OUT / "best.pt",
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint["model"])
model.eval()

# Compare inference methods on validation, never hidden labels.
raw_correct = 0
flip_correct = 0
val_total = 0
raw_predictions = []
flip_predictions = []

with torch.inference_mode():
    for x, y in val_dl:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            raw_scores = model(x)
            flip_scores = (
                raw_scores + model(torch.flip(x, dims=[3]))
            ) / 2

        raw_pred = raw_scores.argmax(1)
        flip_pred = flip_scores.argmax(1)
        raw_correct += (raw_pred == y).sum().item()
        flip_correct += (flip_pred == y).sum().item()
        val_total += len(y)
        raw_predictions.extend(raw_pred.cpu().tolist())
        flip_predictions.extend(flip_pred.cpu().tolist())

raw_acc = raw_correct / val_total
flip_acc = flip_correct / val_total
use_flip = flip_acc > raw_acc

pd.DataFrame({
    "id": train.iloc[va].id.to_numpy(),
    "label": labels[va],
    "prediction_raw": raw_predictions,
    "prediction_flip": flip_predictions,
}).to_csv(OUT / "validation_predictions.csv", index=False)

config["selected_horizontal_flip_averaging"] = use_flip
(OUT / "config.json").write_text(json.dumps(config, indent=2))
checkpoint["config"] = config
torch.save(checkpoint, OUT / "best.pt")

print(
    f"Validation: raw={raw_acc:.4f}, flip={flip_acc:.4f}; "
    f"using flip averaging={use_flip}",
    flush=True,
)

# Generate predictions in test.csv order.
predictions = []
with torch.inference_mode():
    for x in test_dl:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            scores = model(x)
            if use_flip:
                scores = (
                    scores + model(torch.flip(x, dims=[3]))
                ) / 2
        predictions.extend(scores.argmax(1).cpu().tolist())

assert len(predictions) == len(test)
lookup = pd.Series(predictions, index=test.id)
submission = sample[["id"]].copy()
submission["label"] = submission.id.map(lookup)

assert submission.label.notna().all()
submission["label"] = submission.label.astype(int)
assert submission.columns.tolist() == ["id", "label"]
assert submission.id.tolist() == sample.id.tolist()
assert submission.id.is_unique
assert submission.label.between(0, 9).all()

submission.to_csv(OUT / "submission.csv", index=False)

summary = {
    "best_validation_accuracy": best,
    "best_epoch": best_epoch,
    "validation_accuracy_flip": flip_acc,
    "selected_inference_validation_accuracy": max(raw_acc, flip_acc),
    "use_horizontal_flip_averaging": use_flip,
    "trainable_parameters": params,
    "submission_rows": len(submission),
    "submission_sha256": hashlib.sha256(
        (OUT / "submission.csv").read_bytes()
    ).hexdigest(),
}
(OUT / "results.json").write_text(json.dumps(summary, indent=2))

print("Training complete.", flush=True)
print("Best raw validation:", best, "| Best epoch:", best_epoch)
print("Selected inference validation:", max(raw_acc, flip_acc))
print("Saved submission.csv:", submission.shape)
print("Output folder:", OUT)
print(submission.head())

GPU: Tesla T4
Data: /kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl
Output folder: /kaggle/working/resnet_3stage_mixup_20261030_065323_664850
Loading permitted images into memory...
Trainable parameters: 4327754
Epoch 1/200: loss=2.1797 mixup_train=0.2439 val=0.3794 best=0.3794 seconds=36
Epoch 2/200: loss=1.8125 mixup_train=0.3963 val=0.4960 best=0.4960 seconds=19
Epoch 3/200: loss=1.6766 mixup_train=0.4731 val=0.5086 best=0.5086 seconds=20
Epoch 4/200: loss=1.5592 mixup_train=0.5405 val=0.5960 best=0.5960 seconds=22
Epoch 5/200: loss=1.4780 mixup_train=0.5811 val=0.6432 best=0.6432 seconds=21
Epoch 6/200: loss=1.4094 mixup_train=0.6156 val=0.6824 best=0.6824 seconds=20
Epoch 7/200: loss=1.3559 mixup_train=0.6441 val=0.7324 best=0.7324 seconds=21
Epoch 8/200: loss=1.2623 mixup_train=0.6850 val=0.7682 best=0.7682 seconds=21
Epoch 9/200: loss=1.2845 mixup_train=0.6761 val=0.7346 best=0.7682 seconds=21
Epoch 10/200: loss=1.2346 mixup_train=0.7035 val=0.7834 best=0.7